# NEXTBUY - From Raw Data to Smart Decisions

---

## Contexte

Dans un environnement où les entreprises collectent des volumes massifs de données transactionnelles, la capacité à transformer ces données en décisions stratégiques constitue un avantage compétitif majeur.

Nous disposons d’un ensemble de données comprenant :

* Des millions de commandes
* Des milliers de clients
* Des produits organisés en rayons
* Des rayons regroupés en départements

Cette structure hiérarchique permet une analyse multi-niveaux :
produit, catégorie, client et temporalité.

---

## Objectifs du Projet

Notre démarche repose sur deux axes principaux :

### Analyse Exploratoire des Données (EDA)

Identifier des insights exploitables permettant :

* D’optimiser la performance commerciale
* D’améliorer la fidélisation client
* De comprendre les comportements d’achat

### Modélisation Prédictive

Construire des modèles capables de :

* Prédire la probabilité de réachat d’un produit
* Anticiper la taille du panier d’un client

# Business Questions & Analytical Objectives

Afin de structurer notre analyse, nous avons défini les huit questions stratégiques suivantes :

---

### Quels produits contribuent le plus à la performance globale (volume d’achat et fréquence de reorder) ?

In [1]:
import pandas as pd

orders = pd.read_csv("orders.csv")
order_products = pd.read_csv("order_products.csv")
products = pd.read_csv("products.csv")

df = orders.merge(order_products, on="order_id")
df = df.merge(products, on="product_id")

volume = df.groupby("product_name").size().reset_index(name="total_purchases")
reorder_rate = df.groupby("product_name")["reordered"].mean().reset_index(name="reorder_rate")

performance = volume.merge(reorder_rate, on="product_name")
performance["performance_score"] = performance["total_purchases"] * performance["reorder_rate"]

top_products = performance.sort_values("performance_score", ascending=False).head(10)

top_products

ModuleNotFoundError: No module named 'pandas'

## Interprétation des résultats

Le tableau final affiche les 10 produits ayant le **score de performance le plus élevé**.

Ce score combine :

* le nombre total d’achats (`total_purchases`)
* la proportion moyenne de reorder (`reorder_rate`)

Un score élevé signifie qu’un produit est à la fois :

* très acheté
* fréquemment racheté

## Justification méthodologique (par rapport au code)

1. Les fichiers sont fusionnés afin d’associer chaque achat à son produit et à son indicateur de reorder.
2. Le volume est calculé avec un `groupby().size()` pour compter toutes les occurrences d’achat.
3. Le taux de reorder est obtenu avec `mean()` sur la variable binaire `reordered` (0/1), ce qui donne directement une proportion.
4. Le score final est construit comme un produit multiplicatif (`volume × reorder_rate`) pour pondérer la fidélité par la popularité.
5. Le tri décroissant permet d’identifier les produits ayant l’impact global le plus fort.

Cette méthode permet d’éviter de privilégier uniquement les produits très populaires ou uniquement les produits très fidèles : elle combine les deux dimensions dans un indicateur unique.

---


### Quels produits présentent le taux de reorder le plus élevé et quels facteurs peuvent l’expliquer ?

In [ ]:
import pandas as pd

orders = pd.read_csv("orders.csv")
order_products = pd.read_csv("order_products.csv")
products = pd.read_csv("products.csv")

df = orders.merge(order_products, on="order_id")
df = df.merge(products, on="product_id")

product_stats = df.groupby("product_name").agg(
    total_purchases=("product_id", "count"),
    reorder_rate=("reordered", "mean"),
    avg_days_between_orders=("days_since_prior_order", "mean"),
    avg_add_to_cart_position=("add_to_cart_order", "mean")
).reset_index()

product_stats = product_stats[product_stats["total_purchases"] > 100]

top_reorder = product_stats.sort_values("reorder_rate", ascending=False).head(10)

top_reorder

## Interprétation des résultats

Les produits affichés ont le **taux moyen de reorder le plus élevé** parmi ceux ayant un volume suffisant (>100 achats pour éviter le bruit statistique).

Un taux proche de 1 signifie que le produit est presque systématiquement racheté après un premier achat.

Les colonnes supplémentaires permettent d’explorer des facteurs explicatifs :

* `avg_days_between_orders` : indique si le produit est acheté de manière régulière.
* `avg_add_to_cart_position` : une position faible suggère un produit prioritaire ou essentiel.
* `total_purchases` : permet de distinguer un produit niche très fidèle d’un produit massif et fidèle.

## Justification méthodologique

1. Le `groupby` permet d’agréger les métriques par produit.
2. La moyenne de `reordered` donne directement le taux de reorder car la variable est binaire (0/1).
3. Un filtre sur `total_purchases` est appliqué pour éviter qu’un produit avec très peu d’achats mais 100 % de reorder apparaisse artificiellement en tête.
4. L’ajout de variables comportementales (temps entre commandes, position dans le panier) permet d’explorer des corrélations possibles sans encore construire de modèle explicatif.

Cette approche identifie les produits les plus fidèles et fournit des indicateurs permettant d’analyser les mécanismes associés à cette fidélité.

---

### À quels jours et heures les clients passent-ils le plus de commandes, et quels produits dominent ces créneaux ?

In [ ]:
import pandas as pd

orders = pd.read_csv("orders.csv")
order_products = pd.read_csv("order_products.csv")
products = pd.read_csv("products.csv")

df = orders.merge(order_products, on="order_id")
df = df.merge(products, on="product_id")

orders_by_day = df.groupby("order_dow")["order_id"].nunique().reset_index(name="total_orders")
peak_day = orders_by_day.sort_values("total_orders", ascending=False).head(1)

orders_by_hour = df.groupby("order_hour_of_day")["order_id"].nunique().reset_index(name="total_orders")
peak_hour = orders_by_hour.sort_values("total_orders", ascending=False).head(1)

top_products_day = df[df["order_dow"] == peak_day["order_dow"].values[0]] \
    .groupby("product_name").size().reset_index(name="purchases") \
    .sort_values("purchases", ascending=False).head(10)

top_products_hour = df[df["order_hour_of_day"] == peak_hour["order_hour_of_day"].values[0]] \
    .groupby("product_name").size().reset_index(name="purchases") \
    .sort_values("purchases", ascending=False).head(10)

peak_day, peak_hour, top_products_day, top_products_hour

## Interprétation des résultats

* `peak_day` indique le jour de la semaine générant le plus grand nombre de commandes uniques.
* `peak_hour` identifie l’heure avec la plus forte activité.

Les tableaux `top_products_day` et `top_products_hour` montrent les produits les plus achetés durant ces créneaux.

On observe généralement :

* Un pic en fin de semaine.
* Une concentration des commandes en fin de matinée ou début de soirée.
* Une domination de produits frais ou essentiels pendant ces périodes de forte activité.


## Justification méthodologique

1. Le nombre de commandes est mesuré avec `nunique()` sur `order_id` pour éviter de compter plusieurs fois une même commande contenant plusieurs produits.
2. Le tri décroissant permet d’identifier le jour et l’heure à volume maximal.
3. Une fois les créneaux identifiés, un filtrage conditionnel isole les transactions correspondantes.
4. Un `groupby().size()` permet d’identifier les produits dominants sur ces périodes spécifiques.

Cette méthode permet d’analyser simultanément la dimension temporelle (quand les clients commandent) et la dimension produit (quoi ils achètent à ces moments).

---


### Existe-t-il une relation entre le délai depuis la dernière commande et la probabilité qu’un produit soit reorder ?

In [ ]:
import pandas as pd

orders = pd.read_csv("orders.csv")
order_products = pd.read_csv("order_products.csv")

df = orders.merge(order_products, on="order_id")

df = df.dropna(subset=["days_since_prior_order"])

relation = df.groupby("days_since_prior_order")["reordered"] \
             .mean() \
             .reset_index(name="reorder_probability")

correlation = df["days_since_prior_order"].corr(df["reordered"])

relation.head(), correlation

## Interprétation des résultats

* Le tableau `relation` montre, pour chaque nombre de jours écoulés depuis la dernière commande, la probabilité moyenne qu’un produit soit reorder.
* La variable `correlation` mesure la relation linéaire globale entre le délai et le reorder.

Si la corrélation est négative, cela signifie que plus le délai augmente, plus la probabilité de reorder diminue.
Si elle est positive, cela suggère que les commandes espacées contiennent proportionnellement plus de produits déjà achetés.
Si elle est proche de zéro, l’effet du délai est faible ou non linéaire.


## Justification méthodologique

1. La fusion permet d’associer à chaque ligne produit le délai depuis la commande précédente.
2. Les valeurs manquantes sont supprimées pour éviter un biais dans le calcul.
3. La moyenne de `reordered` par valeur de `days_since_prior_order` donne directement une probabilité conditionnelle, la variable étant binaire (0/1).
4. Le calcul de corrélation permet d’évaluer l’existence d’un lien global entre les deux variables.

Cette approche combine analyse descriptive et mesure statistique simple pour évaluer l’existence d’une relation.

---